In [1]:
import numpy as np
import polars as pl
import polars.selectors as pls
import pandas as pd
import seaborn as sea

import catboost, lightgbm, xgboost

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import balanced_accuracy_score
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

import shap
import optuna

In [2]:
def format_pl():
  """FLOAT DISPLAY FORMATTING"""
  pl.Config.set_fmt_float("mixed")
  """STRING FORMATTING"""
  pl.Config.set_fmt_str_lengths(50)
  """TABLE FORMATTING"""
  pl.Config.set_tbl_rows(8)
  pl.Config.set_tbl_cols(30)
  pl.Config.set_tbl_width_chars(200)
  pl.Config.set_tbl_cell_alignment("RIGHT")
  pl.Config.set_tbl_hide_dtype_separator(True)
  pl.Config.set_tbl_hide_column_data_types(True)

format_pl()

In [3]:
class_map = {
    "GALAXY":0,
    "QSO":1,
    "STAR":2
}

# Reading and Processing Data

In [4]:
train = pl.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test  = pl.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")
sample_submission = pl.read_csv("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

In [5]:
sample_submission.head()

id,class
577347,"""GALAXY"""
577348,"""GALAXY"""
577349,"""GALAXY"""
577350,"""GALAXY"""
577351,"""GALAXY"""


In [6]:
train.head(3)

id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,"""M""","""Red_Sequence""","""GALAXY"""
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,"""M""","""Red_Sequence""","""GALAXY"""
2,179.792648,35.344843,21.035203,21.079128,21.17184,20.582629,20.557366,2.82377,"""O/B""","""Blue_Cloud""","""QSO"""


In [7]:
test.head(3)

id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
577347,120.719779,23.924249,23.668066,21.95168,21.086183,20.180032,19.202124,0.429042,"""G/K""","""Red_Sequence"""
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.86033,19.687691,0.867305,"""M""","""Red_Sequence"""
577349,173.568731,-1.7564,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,"""G/K""","""Blue_Cloud"""


In [8]:
train_df = train.with_columns(
    ## Subtraction features
    (pl.col("u") - pl.col("g")).alias("u_minus_g"),
    (pl.col("g") - pl.col("r")).alias("g_minus_r"),
    (pl.col("r") - pl.col("i")).alias("r_minus_i"),
    (pl.col("r") - pl.col("z")).alias("r_minus_z"),
    ## Avg feature
    ((pl.col("u") + pl.col("g") + pl.col("r") + pl.col("i") + pl.col("z"))/5).alias("avg_bands"),
    ## String concatenation
    pl.concat_str(["spectral_type", "galaxy_population"], separator="_").alias("spectral_population_concat"),
    ##
    (pl.col("alpha").radians()).alias("alpha_rad"),
    (pl.col("delta").radians()).alias("delta_rad"),
    
).with_columns(
    ## Cartesian coordinates
    (pl.col("delta_rad").cos() * pl.col("alpha_rad").cos()).alias("sky_x"),
    (pl.col("delta_rad").cos() * pl.col("alpha_rad").sin()).alias("sky_y"),
    pl.col("delta_rad").sin().alias("sky_z")
).drop("id")

In [9]:
train_df = train_df.to_pandas()
train_df["class"] = train_df["class"].map(class_map)

In [10]:
train_df["spectral_type"] = train_df["spectral_type"].astype("category")
train_df["galaxy_population"] = train_df["galaxy_population"].astype("category")
train_df["spectral_population_concat"] = train_df["spectral_population_concat"].astype("category")

cat_features = ["spectral_type", "galaxy_population", "spectral_population_concat"]

In [11]:
def cross_validate(model, model_type, X, y, cat_features, scorer=balanced_accuracy_score, n_splits=5, k_rounds=1):
    scores = []
    for k in range(k_rounds):
        skfold  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=3126 + k)
        
        for train_idx,val_idx in skfold.split(X, y):
            X_train, y_train = X.iloc[train_idx,:], y[train_idx]
            X_val,   y_val   = X.iloc[val_idx,:],   y[val_idx]

            cloned_model = clone(model)
            ## Fits model
            if model_type=="lgbm":
                cloned_model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    categorical_feature=cat_features,
                    callbacks=[lightgbm.early_stopping(100, verbose=False)])
      
            elif model_type=="xgboost":
                cloned_model.fit(
                    X_train, y_train,
                    eval_set=[(X_val, y_val)],
                    verbose=False)
            
            elif model_type=="catboost":
                pass
    
            else: 
                cloned_model.fit(X_train, y_train)

            ## Stores the score
            scores.append(scorer(y_val, cloned_model.predict(X_val)))
    ## Final array has n_splits*k_rounds entries for scores
    return np.array(scores)

In [12]:
X = train_df.copy()
y = X.pop("class")

# Evaluating LGBM Model

In [13]:
# def objective(trial):
#     params = {
#         ## Tree-based items
#         "max_depth":trial.suggest_int("max_depth", 2, 9),
#         "learning_rate":trial.suggest_float("learning_rate", .05, .5, log=True),
#         "min_child_samples":trial.suggest_int("min_child_samples", 5, 100),
#         ## Column and row sampling
#         "colsample_bytree":trial.suggest_float("colsample_bytree", .5, .9),
#         "subsample_freq":trial.suggest_int("subsample_freq", 1, 10),
#         "subsample":trial.suggest_float("subsample", .5, .9),
#         ## Regularizations
#         "reg_alpha":trial.suggest_float("reg_alpha", 1e-4, 1e-1, log=True),
#         "reg_lambda":trial.suggest_float("reg_lambda", 1e-4, 1e-1, log=True)
#     }
#     params["num_leaves"] = trial.suggest_int("num_leaves", 4, 2**params["max_depth"])
#     params["n_estimators"] = 3000
#     params["random_state"] = 3126
#     params["verbose"] = -1
#     params["n_jobs"] = -1
    
#     model  = LGBMClassifier(**params)
#     scores = cross_validate(model, "lgbm", X, y, cat_features)
#     return np.mean(scores)

# study = optuna.create_study(
#     direction="maximize",
#     sampler=optuna.samplers.TPESampler(seed=3126),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=15)
# )
# study.optimize(objective, n_trials=100)  

In [14]:
## Best parameter set estimated by optuna
lgbm_params = {
    'max_depth': 8, 
    'learning_rate': 0.051888185691814956, 
    'min_child_samples': 8, 
    'colsample_bytree': 0.5781877417128364, 
    'subsample_freq': 2, 
    'subsample': 0.8727678421323111, 
    'reg_alpha': 0.00026846622913149636, 
    'reg_lambda': 0.08072901085548027, 
    'num_leaves': 32
}
lgbm_params["n_estimators"] = 3000
lgbm_params["random_state"] = 3126
lgbm_params["verbose"] = -1
lgbm_params["n_jobs"] = -1

In [15]:
## Balanced accuracy scores of the tuned model
lgbm = LGBMClassifier(**lgbm_params)
cross_validate(lgbm, "lgbm", X, y, cat_features)

array([0.95742859, 0.95859808, 0.95831735, 0.95795469, 0.95769471])